# Example: The Pricing of United States Treasury Coupon-Bearing Notes and Bonds
In this example, we price United States Treasury notes and bonds by discounting their coupon and principal cash flows to the issue/settlement date. The auction determines the coupon rate, yield, and resulting purchase price before settlement.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct coupon-security cash flows:__ Represent the purchase price, periodic coupons, and final principal payment in their correct payment periods.
> * __Price notes and bonds using NPV:__ Discount each promised payment and solve the zero-NPV condition for the security's price.
> * __Compare model and auction prices:__ Apply the valuation model to Treasury data and interpret differences in light of timing and quotation conventions.

We will construct the cash flows and compare the calculated prices with auction observations. Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
# Load the shared L2a paths, course environment, and package imports.
include(joinpath(@__DIR__, "Include.jl"));

  Activating 

project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Data
We'll explore T-note and T-bond prices from United States Treasury auctions since October 2022, downloaded using the [Auction query functionality of TreasuryDirect.gov](https://www.treasurydirect.gov/auctions/auction-query/) and vendored with the course package. 

We load the dataset using [the `MyTreasuryNotesAndBondsDataSet(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/data/#VLQuantitativeFinancePackage.MyTreasuryNotesAndBondsDataSet) exported by the `VLQuantitativeFinancePackage`, and store the auction records in the `dataset::DataFrame` variable. By default it returns original issues only; reopenings carry an existing security's coupon and so trade well away from par, and are available with `reopenings = true`.

The dataset includes a `TIPS` column copied from TreasuryDirect. Inflation-protected securities have `TIPS = "Yes"`, and their `High Yield` is a real yield rather than a nominal one. Pricing inflation-indexed cash flows is out of scope for this example, so we retain only rows with `TIPS = "No"`.

> __Returned data:__ `Price` is the published price per $100~\mathrm{USD}$ of par. For the retained `TIPS = "No"` rows, `High Yield` is the nominal annual auction yield $y$ compounded semiannually, and `Interest Rate` is the annual coupon rate $c$. Both rates are stored as decimals. Unlike the T-bill example, the published `High Yield` already uses the convention expected by the coupon-security NPV model, so no rate conversion is required.

In [2]:
# Load the Treasury note and bond auction records used in this example.
all_securities = MyTreasuryNotesAndBondsDataSet();

# Retain nominal coupon securities using TreasuryDirect's explicit TIPS flag.
dataset = filter(:TIPS => ==("No"), all_securities)

Row,CUSIP,Security Type,Security Term,Auction Date,Issue Date,Maturity Date,Price,High Yield,Interest Rate,TIPS
,String15,String7,String31,String15,String15,String15,Float64,Float64,Float64,String3
1,91282CRJ2,Note,7-Year,08/27/2026,08/31/2026,08/31/2033,99.9287,0.04512,0.045,No
2,91282CRK9,Note,5-Year,08/26/2026,08/31/2026,08/31/2031,99.92,0.04393,0.04375,No
3,91282CRH6,Note,2-Year,08/25/2026,08/31/2026,08/31/2028,99.85,0.04204,0.04125,No
4,912810UX4,Bond,20-Year,08/19/2026,08/31/2026,08/15/2046,99.0213,0.05204,0.05125,No
5,912810UW6,Bond,30-Year,08/13/2026,08/17/2026,08/15/2056,98.627,0.05216,0.05125,No
6,91282CRF0,Note,10-Year,08/12/2026,08/17/2026,08/15/2036,99.5407,0.04683,0.04625,No
7,91282CRG8,Note,3-Year,08/11/2026,08/17/2026,08/15/2029,99.8854,0.04291,0.0425,No
8,91282CRC7,Note,7-Year,07/28/2026,07/31/2026,07/31/2033,99.4165,0.04473,0.04375,No
9,91282CRB9,Note,2-Year,07/27/2026,07/31/2026,07/31/2028,99.8767,0.04315,0.0425,No


Let's store the dimension (number of records) of our treasury auction dataset in the `number_of_records::Int64` variable using [the `nrow(...)` function exported by the DataFrames.jl package](https://dataframes.juliadata.org/stable/lib/functions/#DataAPI.nrow)

In [3]:
# Count the auction records that will be repriced in Task 2.
number_of_records = nrow(dataset)

236

Before we dive into pricing calculations, think about what factors might influence the price of a Treasury security. What role do you think the coupon rate, time to maturity, and prevailing interest rates play in determining a bond's price?

___

## Task 1: Visualize the cash flows and discounting for coupon-bearing Treasury bonds
Unlike zero-coupon Treasury bills, which have only two cash flow events (investors provide funds to the Treasury and receive the face (par) value at maturity), coupon-bearing Treasury securities are more complicated because of the periodic coupon payments. Thus, it's helpful to visualize the cash flow events of notes and bonds. 

We begin by building [an instance of the `DiscreteCompoundingModel` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.DiscreteCompoundingModel) and store this discount model in the `discount_model::DiscreteCompoundingModel` variable:

In [4]:
# Select discrete compounding for all note and bond valuations below.
discount_model = DiscreteCompoundingModel();

Next, let's build an instance of [the `MyUSTreasuryCouponSecurityModel` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.MyUSTreasuryCouponSecurityModel) using [the `build(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.build-Tuple{Type{MyUSTreasuryCouponSecurityModel},%20NamedTuple}). 

> __Example:__ We'll compute the price and cash flow for a `T = 20-yr` bond with coupon rate $c=1.750\%$, nominal annual yield $y=1.850\%$, coupon frequency $n=2~\mathrm{yr}^{-1}$, and face (par) value $V_P=100~\mathrm{USD}$. The package API stores these inputs in the fields `coupon = c`, `rate = y`, and `λ = n`. TreasuryDirect reports a price of $V_B=98.336995~\mathrm{USD}$ per $100~\mathrm{USD}$ of par for this example.

> __Modeling assumption:__ The classroom model treats $T=20$ years as exactly 40 complete semiannual periods. Treasury's auction calculation uses the security's exact dates and day-count rules, so the two prices should be close but need not be identical.

Similar to zero-coupon T-bills, we'll use [the `short-cut` syntax](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#Short-cut-syntax), which relies on the [Julia pipe `|>` operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping) to compute the coupon-bearing price, forward accumulation factors, and discounted cash flows. 

Let's store the result in the `test_bond::MyUSTreasuryCouponSecurityModel` variable. 

In [5]:
# Define the security and market inputs. Rates are stored as decimal fractions.
T = 20.0;     # years remaining to maturity
y = 0.01850;  # nominal annual yield used to value the bond
c = 0.01750;  # annual coupon rate paid by the bond
n = 2;        # coupon payments and yield-compounding periods per year
par = 100.0;  # face value repaid at maturity, in USD

# Build the bond and apply the discrete-compounding pricing model.
test_bond = build(MyUSTreasuryCouponSecurityModel, (
    T = T, rate = y, coupon = c, λ = n, par = par
)) |> discount_model;

Now that we have populated [the `MyUSTreasuryCouponSecurityModel` instance](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.MyUSTreasuryCouponSecurityModel) stored in the `test_bond` variable, we can pull data from `test_bond` and construct a table. 

The package stores the price in `test_bond.price`, the discounted cash flows in `test_bond.cashflow`, and the forward accumulation factors in `test_bond.discount`. We assign these values clear local names before constructing the table.

In [6]:
# Extract the computed values needed for the price check and cash-flow table.
computed_bond_price = test_bond.price; # model price per 100 USD of par
cashflow = test_bond.cashflow;       # time-0 value of each signed cash flow
accumulation = test_bond.discount;   # package field stores forward accumulation factors

### Check: Are the computed and published bond prices similar?
Compare the full-precision classroom-model price with TreasuryDirect's published price. The signed difference is the model price minus the published price, in USD per $100$ of par.

> __Expected result:__ The equal-period model should reproduce the published price within $0.003~\mathrm{USD}$ per $100~\mathrm{USD}$ of par. The small residual reflects the exact-date and day-count details omitted by this classroom model.

So how did we do?

In [7]:
# Compare the model price with the full-precision TreasuryDirect value quoted above.
published_bond_price = 98.336995; # USD per 100 USD of par
bond_price_comparison = DataFrame(
    method = ["Published Treasury price", "Equal-period NPV model"],
    price = [published_bond_price, computed_bond_price],
    difference = [0.0, computed_bond_price - published_bond_price]
);

# Display both prices and the model-minus-published residual.
pretty_table(bond_price_comparison;
    column_labels = ["Method", "Price (USD/100 par)",
        "Difference from published price (USD/100 par)"],
    table_format = TextTableFormat(borders = text_table_borders__simple),
    fit_table_in_display_horizontally = false,
    formatters = [fmt__printf("%.6f", [2, 3])]);

@assert isapprox(published_bond_price, computed_bond_price; atol = 0.003)

=========================== ===================== ================================================
                    Method   Price (USD/100 par)   Difference from published price (USD/100 par) 
=========================== ===================== ================================================
  Published Treasury price             98.336995                                        0.000000
    Equal-period NPV model             98.334649                                       -0.002346
=========================== ===================== ================================================


The table shows the nominal and discounted cash flows in each period. The model supplies the accumulation factor. We take its reciprocal to obtain the discount factor, then multiply the nominal cash flow by the discount factor. The final column reports the running total of the discounted cash flows.

In [8]:
let
    # Initialize the table dimensions and the bond's contractual payments.
    number_of_periods = length(accumulation); # includes the purchase date at index 0
    final_period = number_of_periods - 1;      # financial index of the maturity payment
    coupon_payment = (test_bond.coupon / test_bond.λ) * test_bond.par; # USD per coupon period
    bond_data_table = Array{Any,2}(undef, number_of_periods, 5); # one row per financial index
    cumulative_discounted_cashflow = 0.0; # running time-0 value, in USD

    # Construct and discount the signed cash flow at each financial index.
    for i ∈ 0:final_period

        accumulation_factor = accumulation[i];     # grows time-0 dollars to period-i dollars
        discount_factor = 1 / accumulation_factor; # returns period-i dollars to time 0

        # The buyer pays the price at i = 0, receives coupons afterward, and
        # receives both the final coupon and par value at maturity.
        nominal_cashflow = if i == 0
            -test_bond.price
        elseif i == final_period
            coupon_payment + test_bond.par
        else
            coupon_payment
        end
        discounted_cashflow = discount_factor * nominal_cashflow; # period-i value in time-0 USD
        cumulative_discounted_cashflow += discounted_cashflow;    # include the current row

        # Verify that the explicit calculation matches the package's stored value.
        @assert isapprox(discounted_cashflow, cashflow[i]; atol = 1e-12)

        # Store the completed row. Julia row i + 1 represents financial index i.
        bond_data_table[i+1,1] = i;
        bond_data_table[i+1,2] = accumulation_factor;
        bond_data_table[i+1,3] = nominal_cashflow;
        bond_data_table[i+1,4] = discounted_cashflow;
        bond_data_table[i+1,5] = cumulative_discounted_cashflow;
    end

    # Display every period with a simple text border and no vertical cropping.
    pretty_table(bond_data_table; 
        column_labels=["Period", "Accumulation factor", "Nominal cashflow", "Discounted cashflow", "Cumulative discounted cashflow"], 
        fit_table_in_display_horizontally = false, fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__simple))
end

========= ===================== ================== ===================== =================================
  Period   Accumulation factor   Nominal cashflow   Discounted cashflow   Cumulative discounted cashflow 
========= ===================== ================== ===================== =================================
       0                   1.0           -98.3346              -98.3346                         -98.3346
       1               1.00925              0.875               0.86698                         -97.4677
       2               1.01859              0.875              0.859034                         -96.6086
       3               1.02801              0.875              0.851161                         -95.7575
       4               1.03752              0.875               0.84336                         -94.9141
       5               1.04711              0.875               0.83563                         -94.0785
       6                1.0568              0.875 

___

## Task 2: Reproduce Published Note and Bond Auction Prices
Next, we price every nominal original-issue note and bond in the `dataset::DataFrame` and compare the classroom-model result with TreasuryDirect's published auction price. The dataset supplies the three quantities needed for this comparison:

* `High Yield` is the nominal annual auction yield $y$, compounded semiannually. It is used directly as the model's `rate`.
* `Interest Rate` is the annual coupon rate $c$. It determines each semiannual coupon and is used as the model's `coupon`.
* `Price` is the published auction price per $100~\mathrm{USD}$ of par. It is the baseline for the pricing error.

Unlike the T-bill calculation, no rate conversion is needed: the published `High Yield` already matches the coupon-security NPV convention. For each record, `securityterm` converts the whole-year term to $T$, and we use $n=2~\mathrm{yr}^{-1}$. The classroom model therefore treats the security as $nT$ complete semiannual periods from settlement to maturity; exact Treasury calculations can include additional date and day-count details.

We retain every result in `computed_price_table::DataFrame`, store each populated model in `security_dictionary::Dict{Int64,MyUSTreasuryCouponSecurityModel}`, and display compact error summaries rather than all auction rows.

In [9]:
security_dictionary, computed_price_table, term_error_summary, largest_residuals = let
    # Initialize one table row and one priced model for each auction record.
    computed_price_table = DataFrame(); # published prices, model prices, and errors
    security_dictionary = Dict{Int64,MyUSTreasuryCouponSecurityModel}(); # row index => model
    n = 2; # Treasury notes and bonds pay coupons semiannually

    # Reprice every nominal coupon security in the filtered dataset.
    for i ∈ 1:number_of_records

        # Read the contract terms and auction yield from row i.
        term_label = String(dataset[i, Symbol("Security Term")]); # e.g., "10-Year"
        T = securityterm(term_label); # convert the label to maturity in years
        y = dataset[i, Symbol("High Yield")]; # auction yield as a decimal fraction
        c = dataset[i, Symbol("Interest Rate")]; # annual coupon rate as a decimal fraction

        # Build and price a 100 USD par security using semiannual compounding.
        model = build(MyUSTreasuryCouponSecurityModel, (
            par = 100.0, T = T, rate = y, coupon = c, λ = n
        )) |> discount_model;
        
        # Measure the model-minus-published residual relative to the published price.
        published_price = dataset[i, :Price]; # USD per 100 USD of par
        model_price = model.price;             # USD per 100 USD of par
        signed_price_difference = model_price - published_price; # USD per 100 USD of par
        absolute_price_difference = abs(signed_price_difference);
        absolute_relative_error = absolute_price_difference / published_price;

        # Assemble a reader-facing row; rates are converted from decimals to percent.
        results_tuple = (
            CUSIP = dataset[i, :CUSIP],
            term = dataset[i, Symbol("Security Term")],
            auction_yield_percent = 100*y,
            coupon_rate_percent = 100*c,
            model_price_USD_per_100_par = model_price,
            published_price_USD_per_100_par = published_price,
            signed_price_difference_USD_per_100_par = signed_price_difference,
            absolute_price_difference_USD_per_100_par = absolute_price_difference,
            absolute_relative_error = absolute_relative_error
        );

        # Store the summary row and its full priced model under the same row index.
        push!(computed_price_table, results_tuple);
        security_dictionary[i] = model;
    end

    # Summarize errors by original term and identify the five largest residuals.
    term_error_summary = combine(groupby(computed_price_table, :term),
        nrow => :auctions,
        :absolute_price_difference_USD_per_100_par => mean =>
            :mean_absolute_difference_USD_per_100_par,
        :absolute_price_difference_USD_per_100_par => maximum =>
            :maximum_absolute_difference_USD_per_100_par,
        :absolute_relative_error => (values -> 100*maximum(values)) =>
            :maximum_relative_error_percent
    );
    sort!(term_error_summary, :term);

    ranked_results = sort(computed_price_table,
        :absolute_price_difference_USD_per_100_par; rev = true);
    largest_residuals = first(ranked_results, min(5, nrow(ranked_results))) |> table ->
        select(table, :CUSIP, :term, :auction_yield_percent, :coupon_rate_percent,
            :model_price_USD_per_100_par, :published_price_USD_per_100_par,
            :signed_price_difference_USD_per_100_par);

    (security_dictionary, computed_price_table, term_error_summary, largest_residuals)
end;

### Compare Pricing Errors Across Security Terms
The table below groups the auction records by original term. The two dollar-error columns report the mean and maximum absolute difference between the classroom-model price and the published Treasury price, measured in USD per $100$ of par. The final column reports the maximum absolute relative error for each term.

Display the term-level summary.

In [10]:
# Display the pricing-error summary for each original security term.
pretty_table(term_error_summary;
    column_labels = ["Term", "Auctions", "Mean |difference| (USD/100 par)",
        "Maximum |difference| (USD/100 par)", "Maximum relative error (%)"],
    table_format = TextTableFormat(borders = text_table_borders__simple),
    fit_table_in_display_horizontally = false,
    formatters = [fmt__printf("%.6f", [3, 4, 5])])

========== ========== ================================= ==================================== =============================
     Term   Auctions   Mean |difference| (USD/100 par)   Maximum |difference| (USD/100 par)   Maximum relative error (%) 
========== ========== ================================= ==================================== =============================
  10-Year         16                          0.000102                             0.000817                     0.000818
   2-Year         47                          0.000045                             0.000365                     0.000365
  20-Year         16                          0.003008                             0.005114                     0.005114
   3-Year         47                          0.000053                             0.000411                     0.000411
  30-Year         16                          0.000145                             0.000631                     0.000643
   5-Year         47       

### Inspect the Largest Pricing Residuals
The next table shows the five securities with the largest absolute price differences. The `Difference` term is the model price minus the published price, so a positive value means the classroom model produces the higher price.

Display the five largest residuals.

In [11]:
# Display the five largest model-minus-published pricing residuals.
pretty_table(largest_residuals;
    column_labels = ["CUSIP", "Term", "Auction yield (%)", "Coupon rate (%)",
        "Model price", "Published price", "Difference (USD/100 par)"],
    table_format = TextTableFormat(borders = text_table_borders__simple),
    fit_table_in_display_horizontally = false,
    formatters = [fmt__printf("%.4f", [3, 4]),
        fmt__printf("%.6f", [5, 6, 7])])

============ ========= =================== ================= ============= ================= ===========================
      CUSIP      Term   Auction yield (%)   Coupon rate (%)   Model price   Published price   Difference (USD/100 par) 
============ ========= =================== ================= ============= ================= ===========================
  912810UN6   20-Year              4.8760            4.8750     99.987317         99.982203                   0.005114
  912810UL0   20-Year              5.0470            5.0000     99.412413         99.407798                   0.004615
  912810UB2   20-Year              4.6350            4.6250     99.870539         99.866542                   0.003997
  912810UX4   20-Year              5.2040            5.1250     99.025258         99.021288                   0.003970
  912810TW8   20-Year              4.7800            4.7500     99.616386         99.612655                   0.003731
============ ========= =================== 

### Check: How well do we estimate the price of notes and bonds at auction?
Let's specify a tolerance and compute the fraction of notes and bonds with an absolute relative error _less than or equal_ to that tolerance. The published auction price is the reference value in the denominator.
* Let `tolerance = 1e-3`, corresponding to $0.1\%$. You can specify a different value by changing `tolerance`.
* Iterate through `computed_price_table` and increment `counter` when `absolute_relative_error` $\leq$ `tolerance`.
* Divide the passing count by the total number of repriced securities.

Let's see how many instruments clear the tolerance.

In [12]:
# Set the largest acceptable relative pricing error: 10⁻³ = 0.1%.
tolerance = 1e-3;
counter = 0; # number of auction records that satisfy the tolerance

# Count records whose model price is sufficiently close to the auction price.
for i ∈ 1:number_of_records
    if computed_price_table[i, :absolute_relative_error] ≤ tolerance
        counter += 1;
    end
end

# Convert the count to the fraction of all repriced securities.
fraction = counter / number_of_records;
println("Fraction satisfying the error tolerance: $(fraction) \
    ($(counter) out of $(number_of_records))");

Fraction satisfying the error tolerance: 1.0 (236 out of 236)


### Discussion question
1. Based on the fraction of instruments that meet our error tolerance, what can you conclude about the effectiveness of the NPV pricing model for Treasury securities? What factors might explain any pricing discrepancies you observe?
2. Which security has the largest residual? Is the classroom-model price above or below the published price, and which timing simplification might explain the difference?
3. What other factors should investors consider when evaluating Treasury securities? How might liquidity, credit risk, and inflation expectations influence investment decisions?

___

## Summary

This example priced coupon-bearing Treasury notes and bonds by placing every promised cash flow in its payment period and discounting those cash flows to the issue/settlement date.

> __Key Takeaways:__
>
> * __Cash-flow timing determines value:__ A coupon security combines periodic coupon payments with the repayment of principal at maturity, and each payment must be discounted over its own time interval.
> * __Price follows from the zero-NPV condition:__ The model price is the purchase-date amount that makes the net present value of the purchase outflow and promised inflows equal to zero under the stated yield convention.
> * __Auction comparisons require consistent conventions:__ Differences between calculated and reported prices can reflect accrued interest, settlement timing, day-count rules, quotations, rounding, or an incorrect model input.

The NPV framework provides a reproducible valuation benchmark, but its comparison with a market quotation is meaningful only when the cash flows, dates, and quotation conventions are aligned.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance.  Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.